In [1]:
## Load required keys from .env
import os
from dotenv import load_dotenv
_ = load_dotenv('../.env')

SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY")
PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY")
BASE_URL = os.environ.get("LANGFUSE_BASE_URL")


In [2]:
# imports for working with traces in Langfuse
from langfuse_utils import *
import requests

# Authentication
import base64
auth = base64.b64encode(f"{PUBLIC_KEY}:{SECRET_KEY}".encode()).decode()
H = {"Authorization": f"Basic {auth}"}

In [ ]:
## Filters for Langfuse
# NAME_FILTER = "LangGraph"     # or None


In [3]:
# imports for notebook features
# import ipywidgets as w

In [4]:

start = datetime.fromisoformat("2025-07-25T14:30:00+00:00")
end   = datetime.fromisoformat("2025-07-25T15:30:00+00:00")

params = {
    "fromTimestamp": iso_utc_plus(start),
    "toTimestamp": iso_utc_plus(end),
    "limit": LIMIT,
}
r = requests.get(f"{BASE_URL}/api/public/traces", headers=H, params=params, timeout=20)
print(r.status_code, r.text[:400])


200 {"data":[{"id":"0a3d6a938d216d42d8282ba1f55a7c64","projectId":"wri_lcl","name":"LangGraph","timestamp":"2025-07-25T15:24:34.262Z","environment":"default","tags":[],"bookmarked":false,"release":null,"version":null,"userId":null,"sessionId":null,"public":false,"input":{"messages":[{"content":"","additional_kwargs":{},"response_metadata":{},"type":"human","name":null,"id":null,"example":false},{"cont


## Time slots to fetch

In [5]:
## where to store traces data
trace_root = "traces_data"
os.makedirs(trace_root, exist_ok=True)


In [23]:
# Provide time slots in CENTRAL TIME (CT) 
slots = [
    ("Ana Benavides",         "2025-07-25 09:30"),
    ("Kuang Keng Kuek Ser",   "2025-07-29 07:00"),
    ("Willie",                "2025-07-29 16:00"),
    ("Berton Pakpahan",       "2025-08-01 09:00"),
]

# Set this appropriately
INTERVIEW_LENGTH_IN_MINUTES = 60

In [20]:
for who, ct_start in slots:
    f_iso, t_iso = ct_window_iso(ct_start, INTERVIEW_LENGTH_IN_MINUTES)
    rows = fetch_window(f_iso, t_iso, BASE_URL, H)
    
    # client-side filter to avoid server 400s on unknown params
    # if NAME_FILTER: rows = [r for r in rows if r.get("name") == NAME_FILTER]

    slug = "".join(c.lower() if c.isalnum() else "_" for c in who)
    folder = os.path.join(trace_root, slug)

    save_jsonl(os.path.join(folder, "traces_list.jsonl"), rows)
    save_csv(os.path.join(folder, "traces_summary.csv"), [summarize(r) for r in rows])

    print(f"{who}: {ct_start} CT → "
      f"{datetime.fromisoformat(f_iso).strftime('%H:%M')} .. "
      f"{datetime.fromisoformat(t_iso).strftime('%H:%M')} UTC | {len(rows)} traces")

Ana Benavides (SGF): 2025-07-25 09:30 CT → 14:30 .. 15:30 | 10 traces
Kuang Keng Kuek Ser (Pulitzer): 2025-07-29 07:00 CT → 12:00 .. 13:00 | 15 traces
Willie (Mongabay): 2025-07-29 16:00 CT → 21:00 .. 22:00 | 28 traces
Berton Pakpahan (IFMN): 2025-08-01 09:00 CT → 14:00 .. 15:00 | 0 traces


## List all saved traces

In [24]:
!tree traces_data/

traces_data/
├── ana_benavides__sgf_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
├── berton_pakpahan__ifmn_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
├── kuang_keng_kuek_ser__pulitzer_
│   ├── traces_list.jsonl
│   └── traces_summary.csv
└── willie__mongabay_
    ├── traces_list.jsonl
    └── traces_summary.csv

5 directories, 8 files
